In [ ]:
# import necessary packages
import pandas as pd
import numpy as np
from sklearn import linear_model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score, GroupKFold
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns

# define colors used in this work
paton_hot_pink = '#d00d44'
rosewood_plum = '#9B6B8E'     
rosewood_sage = '#7BA882'     
rosewood_gold = '#D4A574'       
rosewood_slate = '#5B7FA6'     
rosewood_rose = '#C9858C'        
rosewood_charcoal = '#4A4A4A'    
rosewood_cream = '#F5EDE0' 
tol_grey = '#BBBBBB'

# define plotting parameters
sns.set(context='talk', style='ticks',
        color_codes=True, rc={'legend.frameon': False})
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 300
plt.rcParams["font.family"] = 'Avenir'
plt.rcParams.update({'font.size': 24})
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# define path to data
dataset = 'hammett_predictions.csv'

In [ ]:
# read in the data
df = pd.read_csv(dataset)

# remove H from the para substituents so it is not double-counted
df = df[df['nitrile_id'] != 'para_nitrile_032']

In [ ]:
# Define X and Y
db_X = df['cm5_charge'].values.reshape(-1, 1)
db_Y = df['hammett_parameter'].values.reshape(-1, 1)


# Ensure that para and meta for the same substituent end up in the same group
df['group'] = df['nitrile_id'].str.extract(r'nitrile_(\d+)$')[0]
unique_groups = df['group'].unique()

# print(unique_groups)

train_groups, test_groups = train_test_split(unique_groups, test_size=0.2, random_state=2020)

# Create train and test masks
train_mask = df['group'].isin(train_groups)
test_mask = df['group'].isin(test_groups)

# Assign train and test sets
X_train, X_test = db_X[train_mask], db_X[test_mask]
y_train, y_test = db_Y[train_mask], db_Y[test_mask]

# # Check if any values in X_test are also in X_train
common_values = np.intersect1d(X_test, X_train)
print("Common values between X_test and X_train:", common_values)
common_groups = np.intersect1d(train_groups, test_groups)
print("Common groups between train_groups and test_groups:", common_groups)

# Fit a linear regression model and evaluate the performance of the train and test sets
model = LinearRegression()
model.fit(X_train, y_train)

y_pred_train = model.predict(X_train)
print("Train set R^2: ", r2_score(y_train, y_pred_train))
print("Train MAE score: %.4f" % mean_absolute_error(y_train, y_pred_train))

y_pred_test = model.predict(X_test)
print("Test set R^2: ", r2_score(y_test, y_pred_test))
print("Test MAE score: %.4f" % mean_absolute_error(y_test, y_pred_test))

In [ ]:
# Perform a 10-fold cross validation
gkf = GroupKFold(n_splits=10)  # adjust number of folds as needed
scores = cross_val_score(model, db_X, db_Y.ravel(), cv=gkf, groups=df['group'], scoring='r2')
print("Cross-validated R^2 scores:", scores)
avg_r2 = scores.mean()
print("Mean cross-validated R^2 score: %.4f" % scores.mean())
sd_r2 = scores.std()
mae_scores = -cross_val_score(model, db_X, db_Y.ravel(), cv=gkf, groups=df['group'], scoring='neg_mean_absolute_error')
print("Cross-validated MAE scores:", mae_scores)
avg_mae = mae_scores.mean()
print("Mean cross-validated MAE score: %.4f" % avg_mae)
sd_mae = mae_scores.std()
sd_mae_abs = abs(sd_mae)
print("Standard deviation of cross-validated MAE scores: %.4f" % sd_mae_abs)
print("Standard deviation of cross-validated MAE scores: %.4f" % -cross_val_score(model, db_X, db_Y.ravel(), cv=gkf, groups=df['group'], scoring='neg_mean_absolute_error').std())

In [ ]:
# Visualize the data
plt.xlabel("Experimental Hammett Parameter", fontsize=16)
plt.ylabel("Predicted Hammett Parameter", fontsize=16)
plt.axline((0, 0), slope=1, color=tol_grey, linestyle="--")
plt.scatter(y_train, model.predict(X_train), 
            alpha=0.2, label="train", color=rosewood_slate, s=35)
plt.scatter(y_test, model.predict(X_test), 
            color=rosewood_gold, alpha=0.5, label="test",s=35)
plt.xlim(-0.75, 1.5)
plt.xticks(np.linspace(-0.5, 1.5, 5), [f"{tick:.1f}" for tick in np.linspace(-0.5, 1.5, 5)])
plt.ylim(-0.75, 1.5)
plt.yticks(np.linspace(-0.5, 1.5, 5), [f"{tick:.1f}" for tick in np.linspace(-0.5, 1.5, 5)])
legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='Train', markerfacecolor=rosewood_slate, markersize=12, alpha=0.6),
        Line2D([0], [0], marker='o', color='w', label='Test', markerfacecolor=rosewood_gold, markersize=12, alpha=0.6)
]
plt.legend(handles=legend_elements, loc='best', fontsize=16, frameon=True, fancybox=True, shadow=False, borderpad=0.5)
plt.annotate(f'R$^2$ = {avg_r2:.2f} ± {sd_r2:.2f}', xy=(0.05, 0.90), xycoords='axes fraction', fontsize=16)
plt.annotate(f'MAE = {avg_mae:.2f} ± {sd_mae_abs:.2f}', xy=(0.05, 0.83), xycoords='axes fraction', fontsize=16)

plt.savefig('train-test-fig.svg', bbox_inches='tight', dpi=400)

plt.show()